# 04 Clustering - Sample Or Full Dataset

Sample mode reads sample embeddings and writes metrics under `experiments/local_sample/`. Full mode reads the server-scale embedding directory.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd

sys.path.append('..')
from src.full_pipeline import (
    PROCESSED_PATH,
    EMBEDDING_DIR,
    EXPERIMENTS_DIR,
    parquet_row_count,
    sample_processed_path,
    sample_embedding_dir,
    run_full_clustering_for_embeddings,
)
from src.visualize import plot_metrics_comparison


## 1. Clustering Configuration


In [ ]:
RUN_MODE = "sample"  # "sample" for local, "full" for server
ROWS_PER_CATEGORY = 1_000

ACTIVE_PROCESSED_PATH = sample_processed_path(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else PROCESSED_PATH
ACTIVE_EMBEDDING_DIR = sample_embedding_dir(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else EMBEDDING_DIR
ACTIVE_EXPERIMENTS_DIR = EXPERIMENTS_DIR / 'local_sample' if RUN_MODE == "sample" else EXPERIMENTS_DIR

ENCODERS = ['tfidf', 'w2v', 'glove', 'sbert', 'bge']
LABEL_COLUMN = 'category'
KMEANS_BATCH_SIZE = 2_048 if RUN_MODE == "sample" else 16_384
UMAP_PLOT_SAMPLE_SIZE = None if RUN_MODE == "sample" else 5_000
MIN_EXPECTED_ROWS = 100 if RUN_MODE == "sample" else 1_000_000

if not ACTIVE_PROCESSED_PATH.exists():
    raise FileNotFoundError(f"{ACTIVE_PROCESSED_PATH} not found. Run 02_preprocess.ipynb first.")
row_count = parquet_row_count(ACTIVE_PROCESSED_PATH)
if row_count < MIN_EXPECTED_ROWS:
    raise RuntimeError(f"{ACTIVE_PROCESSED_PATH} has only {row_count:,} rows. Run 02_preprocess.ipynb first.")
if UMAP_PLOT_SAMPLE_SIZE is None:
    UMAP_PLOT_SAMPLE_SIZE = row_count

print(f"Run mode: {RUN_MODE}; rows: {row_count:,}; UMAP points: {UMAP_PLOT_SAMPLE_SIZE:,}; embedding dir: {ACTIVE_EMBEDDING_DIR}")


## 2. Run MiniBatchKMeans


In [ ]:
all_metrics = run_full_clustering_for_embeddings(
    processed_path=ACTIVE_PROCESSED_PATH,
    embedding_dir=ACTIVE_EMBEDDING_DIR,
    results_dir=ACTIVE_EXPERIMENTS_DIR / 'clustering',
    figures_dir=ACTIVE_EXPERIMENTS_DIR / 'figures',
    encoders=ENCODERS,
    label_column=LABEL_COLUMN,
    batch_size=KMEANS_BATCH_SIZE,
    umap_plot_sample_size=UMAP_PLOT_SAMPLE_SIZE,
)

display(pd.DataFrame(all_metrics).T)


## 3. Metric Comparison


In [ ]:
if all_metrics:
    plot_metrics_comparison(
        all_metrics,
        filename=str(ACTIVE_EXPERIMENTS_DIR / 'figures' / 'clustering_metrics_comparison.png'),
    )
else:
    print("No clustering metrics generated. Run 03_encode.ipynb first.")
